# 01c — Destriping the EI maps

**What/why:** remove the push-broom stripe from the EI target. **Approach:** post-hoc (not cube-level;
`01b` showed one per-column correction removes most of it) robust **column-offset subtraction** — each
column's offset = median down the column, high-passed with a 100-px median filter, subtracted from every
row. Caveat (see conclusion): the median is background-dominated, so skin is slightly under-corrected,
leaving a faint value-dependent residual — accepted as zero-mean target noise. Runs offline.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

import config

PROJECT_ROOT = Path(config.__file__).resolve().parent
LOCAL        = (PROJECT_ROOT / config.LOCAL_PROCESSED_DIR).resolve()
EI_MAPS_DIR  = LOCAL / "ei_maps"
manifest     = pd.read_csv(LOCAL / "manifest.csv")
DISCLOSURE_SUBJECTS = ["p012", "p019", "p027"]

def destripe_ei(ei, window=100):
    """Robust column-offset subtraction: one offset per column, estimated from the median
    down the column (robust to outliers, background-dominated), high-passed to isolate the
    stripe jitter, subtracted row-wise. Under-corrects skin slightly (see conclusion)."""
    col    = np.median(ei, axis=0)                     # robust column profile (median over rows)
    offset = col - median_filter(col, size=window)     # per-column stripe = high-freq jitter
    return ei - offset[np.newaxis, :]

def streaking(img, mask=None, min_px=20):
    """Streaking metric (01b). If mask given, restrict the column means to masked pixels.
    Measures only the column-MEAN (DC) part of the stripe — it is blind to the row-varying
    value-dependent residual, so trust the eye for that (see conclusion)."""
    if mask is None:
        mu, sample = img.mean(0), img
    else:
        cnt = mask.sum(0)
        mu     = np.where(cnt > min_px, (img * mask).sum(0) / np.maximum(cnt, 1), np.nan)
        sample = img[mask.astype(bool)]
    ref = (mu[:-2] + mu[2:]) / 2
    dev = np.abs(mu[1:-1] - ref)
    rng = np.percentile(sample, 99) - np.percentile(sample, 1)
    return float(np.nanmean(dev) / rng)

print("EI maps present:", EI_MAPS_DIR.exists(), " | images:", len(manifest))

## 1. Visual check — original vs destriped (fixed colour scale)

Original EI · destriped EI · removed stripe (`original − destriped`), fixed colour scale. The removed
panel should be **vertical lines only, no face** — confirming the correction takes the stripe, not the
signal. (The wide scale hides the faint skin residual, quantified in stage 2)

In [ ]:
EI_VMIN, EI_VMAX = -50, 100

fig, axes = plt.subplots(len(DISCLOSURE_SUBJECTS), 3, figsize=(16, 5 * len(DISCLOSURE_SUBJECTS)))
for row, s in enumerate(DISCLOSURE_SUBJECTS):
    ei = np.load(EI_MAPS_DIR / f"{s}_neutral_left.npy")
    d  = destripe_ei(ei)
    panels = [(ei, "original EI", "RdYlGn_r", EI_VMIN, EI_VMAX),
              (d,  "destriped EI", "RdYlGn_r", EI_VMIN, EI_VMAX),
              (ei - d, "removed stripe", "coolwarm", -12, 12)]
    for ax, (img, title, cmap, vmn, vmx) in zip(axes[row], panels):
        im = ax.imshow(img, cmap=cmap, vmin=vmn, vmax=vmx)
        ax.set_title(f"{s} — {title}"); ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Robust column-median destripe (fixed colour scale)", y=1.005)
plt.tight_layout(); plt.show()

## 2. Does it work? Streaking before vs after (all 306)

The streaking metric measured **globally** and **inside the skin region** (`horizontal-median
reference > 15`, so the mask itself is stripe-free), before and after destriping.

In [ ]:
rows = []
for r in manifest.itertuples(index=False):
    ei  = np.load(EI_MAPS_DIR / f"{r.subject_id}_{r.pose}_{r.view}.npy")
    ref = median_filter(ei, size=(1, 51))
    skin = (ref > 15).astype(np.float32)
    if skin.sum() < 5000:
        continue
    d = destripe_ei(ei)
    rows.append(dict(
        glob_before=streaking(ei),        glob_after=streaking(d),
        skin_before=streaking(ei, skin),  skin_after=streaking(d, skin)))
S = pd.DataFrame(rows)

print(f"n = {len(S)}")
print(f"  global streaking:  {S.glob_before.mean():.4f}  ->  {S.glob_after.mean():.4f}"
      f"   ({100*(1 - S.glob_after.mean()/S.glob_before.mean()):.0f}% lower)")
print(f"  skin   streaking:  {S.skin_before.mean():.4f}  ->  {S.skin_after.mean():.4f}"
      f"   ({100*(1 - S.skin_after.mean()/S.skin_before.mean()):.0f}% lower)")

## Conclusion

| streaking (DC part) | before | after | reduction |
|---|---|---|---|
| global | 0.077 | 0.012 | **84%** |
| skin | 0.074 | 0.017 | **77%** |

Removed-stripe panels are vertical lines only (no face) → the erythema signal is untouched.

**A value-dependent residual remains on skin** — the metric sees only the column-mean (DC) part, and
the offset is background-dominated so skin is under-corrected. **We stop here:** the residual varies
along rows, so no per-column method removes it, and every stronger method tested leaks facial structure
into the removed component (corrupting the target). It doesn't matter — masks from this vs the best
alternative agree at **IoU 0.989** (n=40), and the residual is zero-mean with clean RGB input, so the
model averages over it rather than reproducing it.

**Next:** `destripe_ei` → `src/ei_computation.py`, batch-write `ei_maps_destriped/`, point the target there.